# Mean Shift — climbing the density to its modes

> Tutorial pair for [`mean_shift.py`](mean_shift.py).

## 1. Intuition
Put a smooth "bump" (kernel) on every data point and add them up: that is a
**kernel density estimate** of where the data is dense. Now let each point roll
*uphill* on this density surface. Points sliding into the same peak (mode) belong
to the same cluster. You never set $k$ — the number of clusters is whatever the
density's peaks decide, controlled by a single **bandwidth** $h$.

## 2. Concept (the slide)
- **KDE:** $\hat f(x)=\frac{1}{n h^d}\sum_i K\!\big(\tfrac{x-x_i}{h}\big)$.
- **Mean-shift step:** replace $x$ by the **kernel-weighted mean** of nearby
  points. This is gradient ascent on $\hat f$ with an automatic step size.
- **Kernels:** *flat* (average points within radius $h$) or *Gaussian* (smooth
  RBF weights, $h$ acts as a std).
- **Clusters:** seed a trajectory at every point, run to convergence, then merge
  points whose modes nearly coincide. Bandwidth $h$ is the whole story: small $h$
  $\Rightarrow$ many modes, large $h$ $\Rightarrow$ few.

## 3. Math derivation

Write the kernel via a radial **profile** $k$: $K(u)=c_k\,k(\lVert u\rVert^2)$.
The density estimate is
$$\hat f(x)=\frac{c_k}{n h^d}\sum_{i=1}^n k\!\Big(\Big\lVert\frac{x-x_i}{h}\Big\rVert^2\Big).$$

**Gradient.** Differentiate w.r.t. $x$ (let $g=-k'$):
$$\nabla\hat f(x)=\frac{2c_k}{n h^{d+2}}\sum_i (x_i-x)\,k'\!\big(\cdot\big)
=\frac{2c_k}{n h^{d+2}}\sum_i (x_i-x)\,\big(-g(\cdot)\big)
=\frac{2c_k}{n h^{d+2}}\Big[\sum_i (x_i-x)\,g_i\Big],$$
with $g_i=g\big(\lVert(x-x_i)/h\rVert^2\big)$. Factor out $\sum_i g_i$:
$$\nabla\hat f(x)=\underbrace{\frac{2c_k}{n h^{d+2}}\sum_i g_i}_{\ge 0}\;
\Big[\underbrace{\frac{\sum_i x_i\,g_i}{\sum_i g_i}-x}_{\textstyle\equiv\ m(x)}\Big].$$

The bracket is the **mean-shift vector**
$$\boxed{\,m(x)=\frac{\sum_i x_i\,g_i}{\sum_i g_i}-x\,}$$
— it points in the **same direction as the density gradient** but is already
normalized by the local density. Hence the fixed-point update
$$x \leftarrow x + m(x) = \frac{\sum_i x_i\,g_i}{\sum_i g_i}$$
is **adaptive gradient ascent**: a big step in flat regions, a small one near a
peak. Comaniciu & Meer prove the trajectory is smooth and converges to a
stationary point (a mode) of $\hat f$.

**Kernels.**
- *Gaussian:* $k(t)=e^{-t/2}\Rightarrow g(t)=e^{-t/2}$, so weights are
  $g_i=\exp(-\lVert x-x_i\rVert^2/2h^2)$ — a soft RBF average.
- *Flat (Epanechnikov-like step):* $k(t)=\mathbf 1[t\le 1]\Rightarrow g$ is the
  indicator, so the update is simply the **mean of points within radius $h$**.

**Clustering.** Run the iteration from each data point; points whose converged
modes lie within a tolerance are merged into one cluster. **Bandwidth** is chosen
by a heuristic, e.g. the mean of the $q$-quantile nearest-neighbor distances.

## 4. NumPy implementation (flat + Gaussian kernels, bandwidth heuristic)

In [ ]:
# ===== actual implementation from mean_shift.py =====
from __future__ import annotations

import numpy as np

SEED = 0

import torch

def get_device():
    """Pick the best available device: cuda > mps > cpu."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    from sklearn.datasets import make_blobs
    from sklearn.metrics import adjusted_rand_score

    X, ytrue = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=SEED)

    # the flat kernel uses h as a hard radius; the Gaussian kernel uses it as a
    # std, so it needs a smaller h to resolve the same clusters.
    h_flat = estimate_bandwidth(X, quantile=0.3)
    h_gauss = h_flat / 2.5
    print(f"estimated bandwidth: flat h={h_flat:.3f}  gaussian h={h_gauss:.3f}")

    bw = {"gaussian": h_gauss, "flat": h_flat}
    for kern in ("gaussian", "flat"):
        ms = MeanShiftNumPy(bandwidth=bw[kern], kernel=kern).fit(X)
        print(f"NumPy mean-shift ({kern:8s}): clusters={ms.n_clusters_}  "
              f"ARI={adjusted_rand_score(ytrue, ms.labels_):.3f}")

    mt = MeanShiftTorch(bandwidth=h_gauss, kernel="gaussian").fit(X)
    print(f"Torch mean-shift (gaussian): clusters={mt.n_clusters_}  "
          f"ARI={adjusted_rand_score(ytrue, mt.labels_):.3f}  device={mt.device}")

    # bandwidth controls cluster count: small h -> many modes, large h -> few
    print("\nBandwidth vs #clusters (gaussian):")
    for hh in (0.3, 0.5, h_gauss, 1.5, 3.0):
        n = MeanShiftNumPy(bandwidth=hh, kernel="gaussian").fit(X).n_clusters_
        print(f"  h={hh:4.2f}: {n} clusters")


class MeanShiftNumPy:
    r"""
    KDE with kernel profile k:   f(x) = (c / n h^d) sum_i k(||(x - x_i)/h||^2).
    Its gradient points toward higher density; setting it proportional to the
    *mean shift vector*

        m(x) = [ sum_i x_i g(||(x-x_i)/h||^2) / sum_i g(...) ] - x,

    where g = -k'. The fixed-point iteration x <- x + m(x) climbs to a mode.

      - Gaussian kernel:  k(t) = exp(-t/2)  -> g(t) = exp(-t/2) (weights = RBF).
      - Flat kernel:      k(t) = 1[t<=1]    -> g is the indicator; m(x) = mean of
                          neighbors within radius h, minus x.
    """

    def __init__(self, bandwidth=1.0, kernel="gaussian", max_iter=300,
                 tol=1e-4, cluster_eps=None):
        self.bandwidth, self.kernel = bandwidth, kernel
        self.max_iter, self.tol = max_iter, tol
        # modes closer than cluster_eps are merged into one cluster
        self.cluster_eps = cluster_eps if cluster_eps is not None else bandwidth / 2

    def _shift(self, y, X):
        # one mean-shift step for a single point y given all data X
        d2 = ((X - y) ** 2).sum(1)                       # ||x_i - y||^2
        h2 = self.bandwidth ** 2
        if self.kernel == "flat":
            w = (d2 <= h2).astype(float)                 # uniform within radius
        else:                                            # gaussian
            w = np.exp(-0.5 * d2 / h2)
        s = w.sum()
        if s == 0:
            return y
        return (w[:, None] * X).sum(0) / s               # weighted mean (the new y)

    def fit(self, X):
        X = np.asarray(X, float)
        modes = X.copy()                                 # seed a trajectory per point
        for i in range(len(X)):
            y = modes[i]
            for _ in range(self.max_iter):
                y_new = self._shift(y, X)
                if np.linalg.norm(y_new - y) < self.tol:
                    y = y_new; break
                y = y_new
            modes[i] = y

        # merge nearby modes into cluster centers (greedy within cluster_eps)
        centers = []
        labels = np.full(len(X), -1)
        for i, m in enumerate(modes):
            for c, center in enumerate(centers):
                if np.linalg.norm(m - center) < self.cluster_eps:
                    labels[i] = c
                    break
            else:
                labels[i] = len(centers)
                centers.append(m)
        self.cluster_centers_ = np.array(centers)
        self.labels_ = labels
        self.n_clusters_ = len(centers)
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        d2 = ((X[:, None, :] - self.cluster_centers_[None, :, :]) ** 2).sum(2)
        return d2.argmin(1)


def estimate_bandwidth(X, quantile=0.3, n_samples=100, seed=SEED):
    """Median-style bandwidth heuristic: average of the `quantile` nearest distances."""
    X = np.asarray(X, float)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), min(n_samples, len(X)), replace=False)
    d2 = ((X[idx][:, None, :] - X[None, :, :]) ** 2).sum(2)
    d = np.sqrt(np.maximum(d2, 0))
    d.sort(axis=1)
    k = max(1, int(quantile * len(X)))
    return d[:, :k].max(1).mean()

## 5. PyTorch implementation (all seeds shifted in parallel via torch.cdist)

In [ ]:
# ===== actual implementation from mean_shift.py =====
class MeanShiftTorch:
    """All points shifted in parallel; torch.cdist gives the pairwise distances."""

    def __init__(self, bandwidth=1.0, kernel="gaussian", max_iter=300,
                 tol=1e-4, cluster_eps=None, device=None):
        self.bandwidth, self.kernel = bandwidth, kernel
        self.max_iter, self.tol = max_iter, tol
        self.cluster_eps = cluster_eps if cluster_eps is not None else bandwidth / 2
        self.device = device or get_device()

    def fit(self, X):
        Xt = torch.as_tensor(np.asarray(X, np.float32), device=self.device)
        Y = Xt.clone()                                   # one seed per data point
        h2 = self.bandwidth ** 2
        for _ in range(self.max_iter):
            D2 = torch.cdist(Y, Xt) ** 2                 # (n, n) squared distances
            if self.kernel == "flat":
                W = (D2 <= h2).float()
            else:
                W = torch.exp(-0.5 * D2 / h2)
            Wsum = W.sum(1, keepdim=True).clamp_min(1e-12)
            Y_new = (W @ Xt) / Wsum                      # weighted means (all at once)
            if (Y_new - Y).norm(dim=1).max() < self.tol:
                Y = Y_new; break
            Y = Y_new

        modes = Y.cpu().numpy()
        centers, labels = [], np.full(len(modes), -1)
        for i, m in enumerate(modes):
            for c, center in enumerate(centers):
                if np.linalg.norm(m - center) < self.cluster_eps:
                    labels[i] = c; break
            else:
                labels[i] = len(centers); centers.append(m)
        self.cluster_centers_ = np.array(centers)
        self.labels_ = labels
        self.n_clusters_ = len(centers)
        return self

## 6. Train / run — ARI, kernel comparison, bandwidth vs #clusters

In [ ]:
demo()

## 7. Visualization — clusters, mode centers, and convergence trajectories

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import mean_shift as M

X, y = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=0)
h = M.estimate_bandwidth(X, quantile=0.3) / 2.5
ms = M.MeanShiftNumPy(bandwidth=h, kernel="gaussian").fit(X)

# trace a few trajectories rolling uphill to their modes
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X[:, 0], X[:, 1], c=ms.labels_, s=12, cmap="tab10")
ax[0].scatter(ms.cluster_centers_[:, 0], ms.cluster_centers_[:, 1],
              c="k", marker="X", s=180)
ax[0].set_title(f"Mean shift: {ms.n_clusters_} modes (h={h:.2f})")
for i in range(0, len(X), 30):
    y0 = X[i].copy(); path = [y0.copy()]
    for _ in range(60):
        y1 = ms._shift(y0, X)
        path.append(y1.copy())
        if np.linalg.norm(y1 - y0) < 1e-4:
            break
        y0 = y1
    path = np.array(path)
    ax[1].plot(path[:, 0], path[:, 1], "-o", ms=2, lw=1)
ax[1].scatter(X[:, 0], X[:, 1], c="lightgray", s=6, zorder=0)
ax[1].set_title("Trajectories ascending the KDE to modes")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **No $k$**: clusters emerge from the density; **bandwidth $h$ is the only knob**
  and it sets everything (small $h\Rightarrow$ over-segment, large $h\Rightarrow$
  one blob). Choose it with a nearest-neighbor heuristic.
- A flat kernel uses $h$ as a hard radius; a Gaussian kernel uses it as a std, so
  the *same* clusters need a smaller $h$ for the Gaussian.
- Robust to cluster shape and outliers (modes ignore sparse points), but the
  naive version is $O(\text{iters}\cdot n^2)$ — seed from a subset / bin for speed.
- Mode-merging tolerance matters: too large fuses real clusters.

**Next:** cluster by the *geometry of a similarity graph* → spectral clustering.